In [2]:
# !pip install langgraph langsmith langchain langchain_groq langchain community


In [71]:
from typing import Annotated
from typing_extensions import TypedDict

In [72]:
# pip install arxiv wikipedia

In [73]:
# !pip install langchain langchain_community

In [74]:
### Working with Tools

from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun

##Arxiv and Wikipedia tools
arxiv_wrapper= ArxivAPIWrapper(top_k_results=1,doc_content_chars_max = 300)
arxiv_tool = ArxivQueryRun(api_wrapper = arxiv_wrapper)
api_wrapper = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max = 300)
wiki_tool = WikipediaQueryRun(api_wrapper = api_wrapper)

In [75]:
wiki_tool.invoke("Who is Elon Musk?")

'Page: Wealth of Elon Musk\nSummary: Elon Musk is the  wealthiest person in the world, with an estimated net worth of US$301 billion as of March 10, 2025, according to the Bloomberg Billionaires Index, and $320 billion according to Forbes, primarily from his ownership stakes in Tesla and SpaceX. \nHavi'

In [76]:
arxiv_tool.invoke("Attention is all you need")

"Published: 2024-07-22\nTitle: Attention Is All You Need But You Don't Need All Of It For Inference of Large Language Models\nAuthors: Georgy Tyukin, Gbetondji J-S Dovonon, Jean Kaddour, Pasquale Minervini\nSummary: The inference demand for LLMs has skyrocketed in recent months, and serving\nmodels with "

In [77]:
tools = [arxiv_tool,wiki_tool]

LangGraph Application

In [78]:
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph,START,END

class State(TypedDict):
  messages:Annotated[list,add_messages]

In [79]:
graph_builder = StateGraph(State)


In [80]:
from langchain_groq import ChatGroq
from google.colab import userdata
groq_api_key = userdata.get('groq_api_key')

In [81]:
llm = ChatGroq(groq_api_key = groq_api_key,model_name="Gemma2-9b-It")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x7fc06f9d60d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7fc06ff9f8d0>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [82]:
llm_with_tools = llm.bind_tools(tools=tools)

In [83]:
def chatbot(state:State):
  return{"messages":[llm_with_tools.invoke(state["messages"])]}

In [84]:
from langgraph.prebuilt import ToolNode,tools_condition

In [85]:
graph_builder.add_node("chatbot",chatbot)
graph_builder.add_edge(START,"chatbot")
tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools",tool_node)
graph_builder.add_conditional_edges("chatbot",tools_condition)
graph_builder.add_edge("tools","chatbot")
graph_builder.add_edge("chatbot",END)

In [90]:
print(graph_builder.nodes)
print(graph_builder.edges)


{'chatbot': StateNodeSpec(runnable=chatbot(tags=None, recurse=True, explode_args=False, func_accepts_config=False, func_accepts={}), metadata=None, input=<class '__main__.State'>, retry_policy=None, ends=()), 'tools': StateNodeSpec(runnable=tools(tags=None, recurse=True, explode_args=False, func_accepts_config=True, func_accepts={'store': ('__pregel_store', None)}, tools_by_name={'arxiv': ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=300)), 'wikipedia': WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/usr/local/lib/python3.11/dist-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=300))}, tool_to

In [86]:
graph = graph_builder.compile()

In [92]:
print(dir(graph))  # List all methods and attributes of the graph object
print(graph.__dict__)  # Check the internal dictionary for attributes



['InputType', 'OutputType', '__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__or__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__ror__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_abatch_with_config', '_abc_impl', '_acall_with_config', '_aprepare_state_snapshot', '_atransform_stream_with_config', '_batch_with_config', '_call_with_config', '_defaults', '_is_protocol', '_prepare_state_snapshot', '_repr_mimebundle_', '_transform_stream_with_config', 'abatch', 'abatch_as_completed', 'aget_graph', 'aget_state', 'aget_state_history', 'aget_subgraphs', 'ainvoke', 'as_tool', 'assign', 'astream', 'astream_events', 'astream_log', 'atransform', 'attach_branch'

In [87]:
from IPython.display import Image,display

try:
  display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
  print("Error drawing graph")

Error drawing graph


In [89]:
# try:
#     mermaid_output = graph.get_graph().draw_mermaid_png()
#     print(mermaid_output)  # Check what this returns
#     display(Image(mermaid_output))
# except Exception as e:
#     print(f"Error drawing graph: {e}")


Error drawing graph: HTTPSConnectionPool(host='mermaid.ink', port=443): Read timed out. (read timeout=10)


In [88]:
user_input = "Hey there! My name is Karan"

events=graph.stream(
    {"messages":[("user",user_input)]},stream_mode="values"
)

for event in events:
  event['messages'][-1].pretty_print()


================================ Human Message =================================

Hey there! My name is Karan
================================== Ai Message ==================================
Tool Calls:
  wikipedia (call_ac41)
 Call ID: call_ac41
  Args:
    query: Karan


/usr/local/lib/python3.11/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.11/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


================================= Tool Message =================================
Name: wikipedia

No good Wikipedia Search Result was found
================================== Ai Message ==================================

Hello Karan! It's nice to meet you.
